# Sketch-of Thought Evaluation
A notebook to evaluate the [Sketch-of-Thought: Efficient LLM Reasoning with Adaptive Cognitive-Inspired Sketching](https://arxiv.org/abs/2503.05179) paper.  
While the code in this notebook doesn't require mandatory hardware acceleration, it is recommended to run in a Colab VM with a GPU, to speed up the execution with LLMs having more than 2 billion parameters.

Clone the official GitHub repo for the paper.

In [ ]:
!git clone https://github.com/SimonAytes/SoT.git
%cd SoT

Remove the version constraint for the HF's Transformers and other libraries indicated in the .toml file for this project, to prevent incompatibility with the Qwen models.

In [ ]:
import re

file_path = "pyproject.toml"

try:
    with open(file_path, "r") as f:
        content = f.read()

    content = re.sub(r'transformers==[\d\.]+', 'transformers', content)
    content = re.sub(r'torch==[\d\.]+', 'torch', content)
    content = re.sub(r'numpy==[\d\.]+', 'numpy', content)
    content = re.sub(r'tokenizers==[\d\.]+', 'tokenizers', content)

    with open(file_path, "w") as f:
        f.write(content)

    print(f"Removed version constraint for some packages in {file_path}")

except FileNotFoundError:
    print(f"Error: {file_path} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Build the SoT package from source.

In [ ]:
!pip install -e .

Initialize the SoT class. The code in the cell below includes the download from the HF's Hub of a pretrained DistilBERT model for automated paradigm selection.

In [ ]:
from sketch_of_thought import SoT

# Initialize SoT
sot = SoT()

Classify a question to determine the best reasoning paradigm.

In [ ]:
question = "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?"
paradigm = sot.classify_question(question)
paradigm

Get the appropriate system prompt for the paradigm.

In [ ]:
system_prompt = sot.get_system_prompt(paradigm)
system_prompt

Get initialized context with examples for the selected paradigm.

In [ ]:
context = sot.get_initialized_context(
    paradigm=paradigm,
    question=question,
    format="llm",
    include_system_prompt=True
)
context

### Example with Transformers

Download the Qwen 2.5 1.5B Instruct model and companion tokenizer.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Format for the input for the model.

In [ ]:
text = tokenizer.apply_chat_template(
    context,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

Generate a response.

In [ ]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

Decode the generated response.

In [ ]:
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response